# multiply-back — faded example 3: Implement both multiply_back functions with Python float coercion

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `multiply-back`. Running the beacon reports progress on the `Backprop: multiply_back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: multiply_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`multiply-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "multiply-back"
DD_SUBTOPIC = "Backprop: multiply_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

When one operand of `x * y` is a Python float, the backward function for the tensor side still works the same way — multiply `grad_out` by the (coerced) scalar. The key is to coerce the float to a tensor using `torch.tensor(y, dtype=grad_out.dtype)` before calling `unbroadcast`, since `unbroadcast` calls `.shape` on the original operand.

## Faded exercise 3

Implement `multiply_back0` and `multiply_back1` that handle the case where either `x` or `y` may be a Python float.

For the float operand: coerce with `torch.tensor(val, dtype=grad_out.dtype)` before the multiply and before passing to `unbroadcast`.

Your task: **fill in the isinstance guard and the coercion for BOTH functions**.

**Fill in:** The isinstance check for the partner operand in each back function, coercing it to a tensor with torch.tensor(val, dtype=grad_out.dtype) when it is a Python float before multiplying with grad_out and calling unbroadcast.

In [ ]:
import torch
from torch import Tensor

def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back0(grad_out: Tensor, out: Tensor, x, y) -> Tensor:
    if not isinstance(y, Tensor):
        y = torch.tensor(y, dtype=grad_out.dtype)
    return unbroadcast(grad_out * y, x if isinstance(x, Tensor) else torch.tensor(x, dtype=grad_out.dtype))

def multiply_back1(grad_out: Tensor, out: Tensor, x, y) -> Tensor:
    if not isinstance(x, Tensor):
        x = torch.tensor(x, dtype=grad_out.dtype)
    return unbroadcast(grad_out * x, y if isinstance(y, Tensor) else torch.tensor(y, dtype=grad_out.dtype))

def _test():
    import torch
    torch.manual_seed(0)
    M = torch.randn(3, 4, requires_grad=True)
    scale = 2.5
    out = M * scale
    out.sum().backward()
    gM = multiply_back0(torch.ones(3,4), out.detach(), M.detach(), scale)
    assert gM.shape == (3, 4)
    assert torch.allclose(gM, M.grad, atol=1e-6)
    gM2 = multiply_back0(torch.ones(3,4), out.detach(), scale, M.detach())
    assert gM2.shape == torch.tensor(scale).shape


def _test():
    import torch
    torch.manual_seed(0)
    M = torch.randn(3, 4, requires_grad=True)
    scale = 2.5
    out = M * scale
    out.sum().backward()
    # back0 for tensor x, float y
    gM = multiply_back0(torch.ones(3,4), out.detach(), M.detach(), scale)
    assert gM.shape == (3, 4), f"shape: {gM.shape}"
    assert torch.allclose(gM, M.grad, atol=1e-6), f"mismatch: {gM - M.grad}"
    # back1 for float x, tensor y
    M2 = torch.randn(2, 5, requires_grad=True)
    out2 = scale * M2
    out2.sum().backward()
    gM2 = multiply_back1(torch.ones(2,5), out2.detach(), scale, M2.detach())
    assert gM2.shape == (2, 5), f"shape: {gM2.shape}"
    assert torch.allclose(gM2, M2.grad, atol=1e-6)
    # Neither crashes when both are tensors
    a = torch.randn(2,2)
    b = torch.randn(2,2)
    g = torch.ones(2,2)
    out3 = a * b
    ga = multiply_back0(g, out3, a, b)
    assert ga.shape == (2,2)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch
from torch import Tensor

def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    for i, (gs, os) in enumerate(zip(grad.shape, original.shape)):
        if os == 1 and gs != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad

def multiply_back0(grad_out: Tensor, out: Tensor, x, y) -> Tensor:
    if not isinstance(y, Tensor):
        y = torch.tensor(y, dtype=grad_out.dtype)
    return unbroadcast(grad_out * y, x if isinstance(x, Tensor) else torch.tensor(x, dtype=grad_out.dtype))

def multiply_back1(grad_out: Tensor, out: Tensor, x, y) -> Tensor:
    if not isinstance(x, Tensor):
        x = torch.tensor(x, dtype=grad_out.dtype)
    return unbroadcast(grad_out * x, y if isinstance(y, Tensor) else torch.tensor(y, dtype=grad_out.dtype))

def _test():
    import torch
    torch.manual_seed(0)
    M = torch.randn(3, 4, requires_grad=True)
    scale = 2.5
    out = M * scale
    out.sum().backward()
    gM = multiply_back0(torch.ones(3,4), out.detach(), M.detach(), scale)
    assert gM.shape == (3, 4)
    assert torch.allclose(gM, M.grad, atol=1e-6)
    gM2 = multiply_back0(torch.ones(3,4), out.detach(), scale, M.detach())
    assert gM2.shape == torch.tensor(scale).shape
```
</details>